# NPT Operations Report — Vaca Muerta & Permian Basin
**Author:** Stefania Dottori | Petroleum Engineer & Data Analyst  
**Dataset:** 40 wells · 1,255 frac stages · 446 NPT events  
**Formations:** Orgánico · La Cocina · Loma Amarilla · Caliza  
**Stack:** Python · pandas · matplotlib · seaborn

---
This notebook reproduces the full NPT analysis from the interactive HTML report.  
Key outputs: NPT drivers, screen-out response optimization, formation risk profiles, and actionable recommendations.

**GitHub Pages:** [View Interactive Dashboard](https://stefaniadottori.github.io/hydraulic-fracturing-analytics-vaca-muerta/)

## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f9f9f9',
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.6,
    'font.family':      'sans-serif',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

PALETTE = {
    'Orgánico':      '#2196F3',
    'La Cocina':     '#FF9800',
    'Loma Amarilla': '#4CAF50',
    'Caliza':        '#9C27B0',
}

# ── Load data ──────────────────────────────────────────────────────────────
DATA_DIR = Path('.')          # adjust if MASTER_dataset.csv is in a subfolder
df = pd.read_csv(DATA_DIR / 'MASTER_dataset.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 1. Dataset Overview

In [ ]:
# ── Stage status summary ───────────────────────────────────────────────────
status_counts = df['stage_status'].value_counts()
print('=== Stage Status ===')
print(status_counts.to_string())

# ── Formation summary ──────────────────────────────────────────────────────
print('\n=== Stages by Formation ===')
print(df['formation'].value_counts().to_string())

# ── NPT events ────────────────────────────────────────────────────────────
npt_df = df[df['event_type'].notna()].copy()
print(f'\nTotal NPT events: {len(npt_df)}')
print(f'Total NPT hours:  {npt_df["npt_hours"].sum():.1f} hrs')

## 2. NPT by Event Type

In [ ]:
# ── Aggregate ─────────────────────────────────────────────────────────────
npt_by_type = (
    npt_df.groupby('event_type')
    .agg(count=('event_type', 'count'), total_hrs=('npt_hours', 'sum'))
    .sort_values('total_hrs', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NPT Analysis by Event Type', fontsize=14, fontweight='bold', y=1.01)

# Count
axes[0].barh(npt_by_type['event_type'], npt_by_type['count'],
             color='#2196F3', edgecolor='white')
axes[0].set_xlabel('Number of Events')
axes[0].set_title('Event Frequency')
axes[0].invert_yaxis()

# Hours
axes[1].barh(npt_by_type['event_type'], npt_by_type['total_hrs'],
             color='#FF5722', edgecolor='white')
axes[1].set_xlabel('Total NPT Hours')
axes[1].set_title('Total Hours Lost')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('fig1_npt_by_event_type.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop NPT drivers:')
print(npt_by_type[['event_type','count','total_hrs']].to_string(index=False))

## 3. NPT by Formation

In [ ]:
npt_by_form = (
    npt_df.groupby('formation')
    .agg(
        events=('event_type', 'count'),
        total_hrs=('npt_hours', 'sum'),
        avg_hrs=('npt_hours', 'mean')
    )
    .sort_values('total_hrs', ascending=False)
    .reset_index()
)

colors = [PALETTE.get(f, '#999') for f in npt_by_form['formation']]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('NPT Distribution by Formation', fontsize=14, fontweight='bold')

bars = axes[0].bar(npt_by_form['formation'], npt_by_form['total_hrs'],
                   color=colors, edgecolor='white', width=0.6)
axes[0].set_ylabel('Total NPT Hours')
axes[0].set_title('Total Hours per Formation')
for bar, val in zip(bars, npt_by_form['total_hrs']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}h', ha='center', va='bottom', fontsize=9)

axes[1].bar(npt_by_form['formation'], npt_by_form['avg_hrs'],
            color=colors, edgecolor='white', width=0.6)
axes[1].set_ylabel('Avg NPT Hours per Event')
axes[1].set_title('Average Event Duration')

plt.tight_layout()
plt.savefig('fig2_npt_by_formation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Stage Duration — Clean vs. NPT Events

In [ ]:
# Classify stages
df['has_npt'] = df['event_type'].notna()

fig, ax = plt.subplots(figsize=(10, 5))

for label, grp, color in [
    ('Clean stages',      df[~df['has_npt']], '#4CAF50'),
    ('Stages with events', df[df['has_npt']],  '#F44336'),
]:
    ax.hist(grp['stage_duration_hrs'].dropna(), bins=40, alpha=0.65,
            label=label, color=color, edgecolor='white')

ax.set_xlabel('Stage Duration (hrs)')
ax.set_ylabel('Count')
ax.set_title('Stage Duration Distribution — Clean vs. NPT Events', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig3_stage_duration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Avg clean stage:       {df[~df['has_npt']]['stage_duration_hrs'].mean():.1f} hrs")
print(f"Avg stage with events: {df[ df['has_npt']]['stage_duration_hrs'].mean():.1f} hrs")

## 5. Screen-Out Rate by Formation

In [ ]:
so_rate = (
    df.groupby('formation')
    .apply(lambda x: (x['stage_status'] == 'Screen-out').sum() / len(x) * 100)
    .reset_index(name='screen_out_pct')
    .sort_values('screen_out_pct', ascending=False)
)

colors = [PALETTE.get(f, '#999') for f in so_rate['formation']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(so_rate['formation'], so_rate['screen_out_pct'],
              color=colors, edgecolor='white', width=0.55)
ax.axhline(so_rate['screen_out_pct'].mean(), color='red',
           linestyle='--', linewidth=1.2, label=f"Portfolio avg: {so_rate['screen_out_pct'].mean():.1f}%")
ax.set_ylabel('Screen-Out Rate (%)')
ax.set_title('Screen-Out Rate by Formation', fontweight='bold')
ax.legend()
for bar, val in zip(bars, so_rate['screen_out_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_screenout_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print(so_rate.to_string(index=False))

## 6. NPT Event Stacked Distribution

In [ ]:
pivot = (
    npt_df.groupby(['formation', 'event_type'])
    .size()
    .unstack(fill_value=0)
)

pivot.plot(
    kind='bar', stacked=True, figsize=(12, 6),
    colormap='tab10', edgecolor='white', linewidth=0.5
)
plt.title('NPT Event Distribution by Formation (Stacked)', fontweight='bold')
plt.xlabel('Formation')
plt.ylabel('Event Count')
plt.xticks(rotation=0)
plt.legend(title='Event Type', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('fig5_stacked_events.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. IP30 by Screen-Out Response

In [ ]:
# Only screen-out stages with production data
so_df = df[
    (df['stage_status'] == 'Screen-out') &
    (df['screenout_response'].notna()) &
    (df['ip30_bopd'].notna())
].copy()

ip_by_response = (
    so_df.groupby('screenout_response')['ip30_bopd']
    .agg(['mean', 'median', 'count'])
    .sort_values('mean', ascending=False)
    .reset_index()
)

response_colors = {'Job terminated': '#4CAF50', 'Rate reduction': '#FF9800',
                   'Flush initiated': '#F44336'}
colors = [response_colors.get(r, '#999') for r in ip_by_response['screenout_response']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(ip_by_response['screenout_response'], ip_by_response['mean'],
              color=colors, edgecolor='white', width=0.55)
ax.set_ylabel('Avg IP30 (bopd)')
ax.set_title('IP30 by Screen-Out Response Strategy', fontweight='bold')
for bar, val, n in zip(bars, ip_by_response['mean'], ip_by_response['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val:.0f} bopd\n(n={n})', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('fig6_ip_by_response.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- IP30 by Screen-Out Response ---')
print(ip_by_response.to_string(index=False))

## 8. Formation Risk Profile Table

In [ ]:
geo_events   = ['NWB Tortuosity', 'Lost Circulation', 'Screen-out']
ops_events   = ['Equipment Delay', 'Pressure Spike', 'Rate Drop', 'Communication']

def geo_npt(g):
    return npt_df[(npt_df['formation'] == g) &
                  (npt_df['event_type'].isin(geo_events))]['npt_hours'].sum()

def ops_npt(g):
    return npt_df[(npt_df['formation'] == g) &
                  (npt_df['event_type'].isin(ops_events))]['npt_hours'].sum()

formations = df['formation'].dropna().unique()

risk_profile = pd.DataFrame({
    'Formation': formations,
    'Geo NPT (hrs)': [geo_npt(f) for f in formations],
    'Ops NPT (hrs)': [ops_npt(f) for f in formations],
    'Screen-Out Rate (%)': [
        (df[df['formation']==f]['stage_status']=='Screen-out').sum() /
        len(df[df['formation']==f]) * 100 for f in formations
    ],
    'Avg IP30 (bopd)': [
        df[df['formation']==f]['ip30_bopd'].mean() for f in formations
    ]
}).sort_values('Geo NPT (hrs)', ascending=False).reset_index(drop=True)

risk_profile = risk_profile.round(1)
print('=== Formation Risk Profile ===')
print(risk_profile.to_string(index=False))
risk_profile

## 9. Top 10 Outlier Wells

In [ ]:
well_npt = (
    npt_df.groupby('well_id')
    .agg(
        total_npt_hrs=('npt_hours', 'sum'),
        event_count=('event_type', 'count'),
        formation=('formation', 'first'),
        operator=('operator', 'first')
    )
    .sort_values('total_npt_hrs', ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
colors = [PALETTE.get(f, '#999') for f in well_npt['formation']]
bars = ax.barh(well_npt['well_id'], well_npt['total_npt_hrs'],
               color=colors, edgecolor='white')
ax.set_xlabel('Total NPT Hours')
ax.set_title('Top 10 Outlier Wells — Highest NPT', fontweight='bold')
ax.invert_yaxis()

# Formation legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in PALETTE.items()]
ax.legend(handles=legend_elements, title='Formation',
          loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('fig7_outlier_wells.png', dpi=150, bbox_inches='tight')
plt.show()

print(well_npt[['well_id','formation','operator','total_npt_hrs','event_count']].to_string(index=False))

## 10. NPT Financial Impact

In [ ]:
COST_PER_HOUR = 20_862   # USD/hr — Vaca Muerta frac crew estimate

total_npt_hrs = npt_df['npt_hours'].sum()
total_cost    = total_npt_hrs * COST_PER_HOUR
saving_10pct  = total_cost * 0.10

print('=== NPT Financial Impact ===')
print(f'Total NPT hours:         {total_npt_hrs:,.1f} hrs')
print(f'Estimated cost @ ${COST_PER_HOUR:,}/hr: ${total_cost:,.0f}')
print(f'10% NPT reduction saves: ${saving_10pct:,.0f}')

# By formation
form_cost = (
    npt_df.groupby('formation')['npt_hours'].sum() * COST_PER_HOUR
).sort_values(ascending=False).reset_index()
form_cost.columns = ['formation', 'estimated_cost_usd']
form_cost['pct_total'] = form_cost['estimated_cost_usd'] / total_cost * 100
print('\n--- Cost by Formation ---')
print(form_cost.round(0).to_string(index=False))

## 11. Key Findings & Recommendations

### Key Findings

| # | Finding | Metric |
|---|---------|--------|
| 1 | **NWB Tortuosity** is the #1 NPT driver by hours | 59.7 hrs total |
| 2 | **Equipment Delay** is #2 | 56.2 hrs total |
| 3 | **La Cocina** has highest geological NPT risk | Clay content 24%, quartz <30% |
| 4 | **Caliza** has the highest screen-out rate | 6.9% |
| 5 | **"Job terminated"** is the optimal screen-out response | Avg 727 bopd IP30 |
| 6 | **"Flush initiated"** yields the worst IP30 | Avg 675 bopd IP30 |
| 7 | 10% NPT reduction = **~$524K recoverable** across this portfolio | 251.6 total hrs |

### Actionable Recommendations

1. **Tortuosity mitigation first** — Deploy near-wellbore diversion or resin-coated proppant in La Cocina and Loma Amarilla to reduce NWB tortuosity events (biggest hours lost).

2. **Equipment redundancy plan** — Equipment Delay (56.2 hrs) is fully preventable. Standardize pre-job equipment checks and on-site spare parts inventory, especially for high-rate Orgánico pads.

3. **Screen-out response protocol** — Stop using Flush as default response. Data shows Job Terminated outperforms Flush by +52 bopd IP30. Implement a decision tree: if SCF > threshold → terminate and re-perforate.

4. **Real-time NPT monitoring** — Integrate Halliburton SENSORI (recently deployed in Vaca Muerta) or equivalent to detect tortuosity signatures and screen-out precursors before they escalate — estimated $1.7M/crew saving potential based on Haynesville benchmark (Corva, 2024).

---
**Stefania Dottori**  
Petroleum Engineer | Data Analyst  
🔗 [GitHub Portfolio](https://github.com/stefaniadottori/hydraulic-fracturing-analytics-vaca-muerta)  
🔗 [Interactive Dashboard](https://stefaniadottori.github.io/hydraulic-fracturing-analytics-vaca-muerta/)